# 🎭 PoetryDuel-GPT — Colab Training

**Train a ~30M-parameter GPT with two-stage training — all genres → Lục Bát fine-tune.**

| Step | Time |
|------|------|
| Clone repo + install | ~2 min |
| Clean + preprocess + tokenize | ~5 min |
| Quick test (200 steps) | ~3 min |
| Stage 1: All genres (15K steps) | ~3 hr on T4, ~2 hr on L4 |
| Stage 2: Lục Bát fine-tune (5K steps) | ~1 hr on T4, ~40 min on L4 |
| Generate poetry | ~1 min |
### Requirements
- Google Colab with GPU runtime (T4 is free, L4 is faster)
- The `poems_dataset.csv` must be uploaded to your Google Drive
  at `MyDrive/poetry-dual-gpt/data/poems_dataset.csv`

### New in this version
- Two-stage training: all genres → Lục Bát fine-tune
- 30M params (n_embd=512, n_layer=8, n_head=8)
- Top-p nucleus sampling, auto-detect genre
- Dual genre: Lục Bát (6→8) + Thất Ngôn (7→7)

In [ ]:
# @title 1. Clone Repo + Install Dependencies
!git clone https://github.com/weseegod/poetry-dual-gpt.git /content/poetry-dual-gpt 2>/dev/null
%cd /content/poetry-dual-gpt
!git pull origin main

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q pandas tokenizers tqdm
!mkdir -p data tokenizer checkpoints

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# @title 2. Mount Drive + Copy Dataset
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# Copy poems_dataset.csv from Drive to Colab VM
DRIVE_CSV = "/content/drive/MyDrive/poetry-dual-gpt/data/poems_dataset.csv"
LOCAL_CSV = "data/poems_dataset.csv"

if os.path.exists(DRIVE_CSV):
    shutil.copy(DRIVE_CSV, LOCAL_CSV)
    size_mb = os.path.getsize(LOCAL_CSV) / 1e6
    print(f"✅ Dataset copied: {size_mb:.1f} MB")
else:
    print(f"⚠️  Dataset not found at {DRIVE_CSV}")
    print("   Upload poems_dataset.csv to your Google Drive at:")
    print("   MyDrive/poetry-dual-gpt/data/poems_dataset.csv")

In [ ]:
# @title 3. Explore Dataset (optional)
%cd /content/poetry-dual-gpt
from src.dataset import explore_dataset
df = explore_dataset()

In [ ]:
# @title 4. Clean Data + Preprocess (Lục Bát & Thất Ngôn) + Train Tokenizer (~5 min)
%cd /content/poetry-dual-gpt

# Clean: filter genres, strip HTML, normalize Unicode, remove duplicates
!python src/clean_data.py

# Preprocess: convert clean CSV → training pairs (strict 6→8 and 7→7)
!python src/preprocess.py

# Train BPE tokenizer on the corpus (includes [THAT_NGON] token)
!python src/train_bpe.py

In [ ]:
# @title 5. Quick Test (200 steps, ~3 min) — verify everything works
%cd /content/poetry-dual-gpt
!python src/train.py --mode test

In [ ]:
# @title 6a. Stage 1 — Pretrain on ALL genres (~3 hr T4, ~2 hr L4)
# Trains on 942K pairs (Lục Bát + Thất Ngôn). Saves checkpoints/best.pt
assert torch.cuda.is_available(), "⚠️  Enable GPU runtime: Runtime → Change runtime type → T4 GPU"

%cd /content/poetry-dual-gpt
!python src/train.py --mode train

# Save Stage 1 checkpoint for Stage 2
!cp checkpoints/final.pt checkpoints/stage1_base.pt
print("✅ Stage 1 complete → checkpoints/stage1_base.pt")

In [ ]:
# @title 6b. Prepare Lục Bát-only corpus (~30 sec)
%cd /content/poetry-dual-gpt

# Filter only [LUC_BAT] pairs from the full corpus
with open('data/poetry_corpus.txt') as f:
    lines = f.readlines()
luc_bat = [l for l in lines if '[LUC_BAT]' in l]
with open('data/corpus_luc_bat.txt', 'w') as f:
    f.writelines(luc_bat)
print(f'Lục Bát pairs: {len(luc_bat):,} / {len(lines):,} total')

In [ ]:
# @title 6c. Stage 2 — Fine-tune on Lục Bát only (~1 hr T4, ~40 min L4)
# Resumes from Stage 1 checkpoint, trains on 651K Lục Bát pairs at lower LR
assert torch.cuda.is_available(), "⚠️  Enable GPU runtime"

%cd /content/poetry-dual-gpt
!python src/train.py \
    --resume checkpoints/stage1_base.pt \
    --corpus data/corpus_luc_bat.txt \
    --steps 5000

print("\n🎉 Two-stage training complete!")
print("   Stage 1 (all genres): checkpoints/stage1_base.pt")
print("   Stage 2 (Lục Bát ft): checkpoints/final.pt")

In [ ]:
# @title 7. Generate Poetry + Evaluate
%cd /content/poetry-dual-gpt

# Compare Stage 1 (all genres) vs Stage 2 (Lục Bát fine-tuned):
print("\n── Stage 1 (all genres) ──")
!python src/sample.py --checkpoint checkpoints/stage1_base.pt --num_samples 2
print("\n── Stage 2 (Lục Bát fine-tuned) ──")
!python src/sample.py --checkpoint checkpoints/final.pt --num_samples 2

In [ ]:
# @title 8. Save Checkpoint to Drive
DRIVE_CKPT = "/content/drive/MyDrive/poetry-dual-gpt/checkpoints"
!mkdir -p {DRIVE_CKPT}
!cp -v checkpoints/final.pt {DRIVE_CKPT}/
!test -f checkpoints/best.pt && cp -v checkpoints/best.pt {DRIVE_CKPT}/ && echo '🏆 Best model saved' || true
print("✅ Checkpoint saved to Drive")

# Save intermediate checkpoints too
import glob
for ckpt in sorted(glob.glob("checkpoints/step_*.pt")):
    !cp -v {ckpt} {DRIVE_CKPT}/
    print(f"✅ {ckpt} saved to Drive")